# Dataset preparation notebook

> This notebook is dedicated to preparing our dataset: combining the images from both DIV2K and FFHQ datasets, then validating inputs and outputs for all of our 4 models (interpolation, SRCNN, Real-ESRGAN and SwinIR).

In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

from PIL import Image

import random
import numpy as np

import os
from pathlib import Path

In [2]:
ARTIFACTS_PATH = Path("../artifacts")
DATA_PATH = Path("../data")

DIV2K_SAMPLES_PATH = DATA_PATH / "div2k_samples"
FFHQ_SAMPLES_PATH = DATA_PATH / "ffhq_samples"

In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    
    print(f"Seed set to: {seed}")


SEED = 42
set_seed(SEED)

Seed set to: 42


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [5]:
class SRDataset(Dataset):
    def __init__(
        self,
        root_dirs: list[Path],
        patch_size: int = 24,    # LR patch size
        upscale_factor: int = 4,
        augment: bool = True,
    ):
        self.patch_size = patch_size
        self.upscale_factor = upscale_factor
        self.augment = augment
        self.hr_patch_size = patch_size * upscale_factor  # default: 96

        self.image_paths = []
        for dir in root_dirs:
            for fname in os.listdir(dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                    fpath = os.path.join(dir, fname)
                    # Skip images that are < HR patch_size
                    w, h = Image.open(fpath).size
                    if w >= self.hr_patch_size and h >= self.hr_patch_size:
                        self.image_paths.append(fpath)

        self.to_tensor = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
        ])


    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        img = Image.open(self.image_paths[index]).convert("YCbCr")
        w, h = img.size

        # Random crop
        x = random.randint(0, w - self.hr_patch_size)
        y = random.randint(0, h - self.hr_patch_size)

        hr_img = img.crop((x, y, x + self.hr_patch_size, y + self.hr_patch_size))
        lr_img = hr_img.resize((self.patch_size, self.patch_size), Image.Resampling.BICUBIC)

        if self.augment:
            # Horizontal flip
            if random.random() > 0.5:
                hr_img = hr_img.transpose(Image.FLIP_LEFT_RIGHT)
                lr_img = lr_img.transpose(Image.FLIP_LEFT_RIGHT)
            # Vertical flip
            if random.random() > 0.5:
                hr_img = hr_img.transpose(Image.FLIP_TOP_BOTTOM)
                lr_img = lr_img.transpose(Image.FLIP_TOP_BOTTOM)
            # 0, 90, 180, 270 degrees rotation
            k = random.randint(0, 3)
            if k > 0:
                hr_img = hr_img.rotate(k * 90)
                lr_img = lr_img.rotate(k * 90)

        # Only Y-channel for SRCNN
        lr_y, _, _ = lr_img.split()
        hr_y, _, _ = hr_img.split()

        return self.to_tensor(lr_y), self.to_tensor(hr_y)


    def __len__(self) -> int:
        return len(self.image_paths)

In [6]:
data = SRDataset(
    root_dirs=[DIV2K_SAMPLES_PATH, FFHQ_SAMPLES_PATH],
    patch_size=24,
    upscale_factor=4,
    augment=True
)

loader = DataLoader(data, batch_size=16, shuffle=True, pin_memory=torch.cuda.is_available())

In [7]:
lr_batch, hr_batch = next(iter(loader))
print(f"LR batch: {lr_batch.shape}")   # [16, 1, 24, 24]
print(f"HR batch: {hr_batch.shape}")   # [16, 1, 96, 96]

LR batch: torch.Size([16, 1, 24, 24])
HR batch: torch.Size([16, 1, 96, 96])


In [8]:
class SRCNN(nn.Module):
    def __init__(self):
        super(SRCNN, self).__init__()
        # Patch extraction
        self.conv1 = nn.Conv2d(1, 64, kernel_size=9, padding=4)
        # Non-linear mapping
        self.conv2 = nn.Conv2d(64, 32, kernel_size=1, padding=0)
        # Reconstruction
        self.conv3 = nn.Conv2d(32, 1, kernel_size=5, padding=2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)
        return x

In [9]:
model = SRCNN()
model

SRCNN(
  (conv1): Conv2d(1, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
  (conv2): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
  (conv3): Conv2d(32, 1, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (relu): ReLU()
)

In [10]:
# LR resize to HR size before SRCNN
lr_upscaled = nn.functional.interpolate(lr_batch, scale_factor=4, mode="bicubic", antialias=True)
print(f"LR upscaled: {lr_upscaled.shape}")  # [16, 1, 96, 96]

LR upscaled: torch.Size([16, 1, 96, 96])


In [11]:
model.eval()
with torch.no_grad():
    output = torch.clamp(model(lr_upscaled), 0.0, 1.0)

print(f"Output shape: {output.shape}")   # [16, 1, 96, 96]
print(f"Output range: [{output.min():.3f}, {output.max():.3f}]")

Output shape: torch.Size([16, 1, 96, 96])
Output range: [0.000, 0.023]


Check if loss can be calculated.

In [12]:
criterion = nn.MSELoss()
loss = criterion(output, hr_batch)
print(f"Loss: {loss.item():.6f}")

Loss: 0.249983
